# Spotify 2023: Audio features across stream levels

I wanted to look at the professor's suggestion of studying the audio features **conditional on stream level** instead of forcing a user-affinity response into the 2023 data.

The question I am looking at here is:

**Do the audio features look different, or relate to each other differently, for songs with different levels of streams?**

I kept the same 8 audio features from the earlier EDA: BPM, danceability, valence, energy, acousticness, instrumentalness, liveness, and speechiness.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from pathlib import Path

# Folder for any tables/figures I may want to use later
output_dir = Path("spotify_option1_outputs")
output_dir.mkdir(exist_ok=True)


## Load the data

If `spotify-2023.csv` is already in the Colab session, this will read it directly. Otherwise Colab will ask me to upload it.


In [ ]:
data_file = Path("spotify-2023.csv")

if not data_file.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        data_file = Path(next(iter(uploaded)))
    except ImportError:
        data_file = Path("/mnt/data/spotify-2023.csv")

df = pd.read_csv(data_file, encoding="latin1")

print("Shape of the original data:", df.shape)
display(df.head())


## Quick check of the variables I need

I only need `streams` and the 8 audio features for this part of the analysis.


In [ ]:
features = [
    "bpm",
    "danceability_%",
    "valence_%",
    "energy_%",
    "acousticness_%",
    "instrumentalness_%",
    "liveness_%",
    "speechiness_%"
]

display(df[["track_name", "artist(s)_name", "streams"] + features].head())


The `streams` column is not completely numeric, so I converted it first. One row could not be converted, so I left that row out of this analysis rather than trying to guess the value.


In [ ]:
df["streams_num"] = pd.to_numeric(df["streams"], errors="coerce")

print("Rows with an invalid stream value:", df["streams_num"].isna().sum())

# Just to see which row caused the problem
display(
    df.loc[df["streams_num"].isna(), ["track_name", "artist(s)_name", "streams"]]
)

df_analysis = df.dropna(subset=["streams_num"]).copy()

print("Number of songs used:", len(df_analysis))


## Make stream-popularity groups

I used **quartiles** here because I wanted a simple low-to-high comparison with enough songs in each group.

- Q1 = lowest 25% of streams
- Q2 = 25% to 50%
- Q3 = 50% to 75%
- Q4 = highest 25% of streams

This gives four equally sized groups.


In [ ]:
quartile_labels = [
    "Q1: Lowest 25%",
    "Q2",
    "Q3",
    "Q4: Highest 25%"
]

df_analysis["stream_quartile"] = pd.qcut(
    df_analysis["streams_num"],
    q=4,
    labels=quartile_labels
)

quartile_summary = (
    df_analysis
    .groupby("stream_quartile", observed=True)["streams_num"]
    .agg(["count", "min", "max"])
)

display(quartile_summary)


## First look: average feature values

Before doing any test, I first compared the average audio-feature values across the four groups. This is just a descriptive check to see where the bigger changes might be.


In [ ]:
feature_means = (
    df_analysis
    .groupby("stream_quartile", observed=True)[features]
    .mean()
    .round(2)
)

display(feature_means)

feature_means.to_csv(output_dir / "feature_means_by_stream_quartile.csv")


From the table, **speechiness** changes the most clearly from the lower-stream groups to the highest-stream group. Danceability is also a little lower in Q4. The other feature averages look more similar across the groups.

I did not want to rely only on means, though, because several of these variables were skewed in the earlier EDA.


## Look at the distributions

I plotted a few of the features that were useful to compare visually. Speechiness and danceability showed the clearest differences in the summary table, while energy and acousticness are included because they had an important relationship in the earlier EDA.


In [ ]:
plot_features = [
    ("speechiness_%", "Speechiness (%)"),
    ("danceability_%", "Danceability (%)"),
    ("energy_%", "Energy (%)"),
    ("acousticness_%", "Acousticness (%)")
]

for feature, ylab in plot_features:
    groups = [
        df_analysis.loc[
            df_analysis["stream_quartile"] == q,
            feature
        ].dropna()
        for q in df_analysis["stream_quartile"].cat.categories
    ]

    plt.figure(figsize=(7, 4.5))
    plt.boxplot(
        groups,
        tick_labels=["Q1\nLowest", "Q2", "Q3", "Q4\nHighest"]
    )
    plt.xlabel("Stream quartile")
    plt.ylabel(ylab)
    plt.title(f"{ylab.replace(' (%)', '')} by stream quartile")
    plt.tight_layout()
    plt.savefig(
        output_dir / f"{feature.replace('%', 'pct')}_by_quartile.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## Kruskal-Wallis test

Since some of the audio features are skewed, I used a **Kruskal-Wallis test** instead of comparing the four groups with a standard one-way ANOVA.

For each feature, the null hypothesis is that its distribution is the same across Q1, Q2, Q3, and Q4.


In [ ]:
results = []

for feature in features:
    groups = [
        df_analysis.loc[
            df_analysis["stream_quartile"] == q,
            feature
        ].dropna()
        for q in df_analysis["stream_quartile"].cat.categories
    ]

    H, p = kruskal(*groups)

    results.append({
        "Feature": feature,
        "H statistic": round(H, 3),
        "p-value": round(p, 4)
    })

kw_results = pd.DataFrame(results).sort_values("p-value")

kw_results["Result at 0.05"] = np.where(
    kw_results["p-value"] < 0.05,
    "Evidence of a difference",
    "No significant evidence"
)

display(kw_results)

kw_results.to_csv(output_dir / "kruskal_wallis_results.csv", index=False)


The smallest p-value is for **speechiness**, followed by **danceability**. At the 0.05 level, those are the two features that show evidence of a difference across the four stream groups.

I am treating these as **exploratory results**. These are the unadjusted p-values from testing all 8 features, so I would be careful about making a strong claim from danceability alone.


## Do the relationships between features change too?

The group means are only one part of the story. I also wanted to see whether the **correlation structure** changes with popularity.

I calculated a separate correlation matrix inside each stream quartile.


In [ ]:
correlation_matrices = {}

for q in df_analysis["stream_quartile"].cat.categories:
    subset = df_analysis.loc[df_analysis["stream_quartile"] == q, features]
    correlation_matrices[q] = subset.corr()

corr_q1 = correlation_matrices["Q1: Lowest 25%"]
corr_q4 = correlation_matrices["Q4: Highest 25%"]

print("Q1: lowest 25%")
display(corr_q1.round(2))

print("Q4: highest 25%")
display(corr_q4.round(2))


I focused on three relationships that were already noticeable in the earlier EDA: energy vs acousticness, danceability vs valence, and energy vs valence.


In [ ]:
relationship_results = []

for q in df_analysis["stream_quartile"].cat.categories:
    corr = correlation_matrices[q]

    relationship_results.append({
        "Stream group": q,
        "Energy vs Acousticness": corr.loc["energy_%", "acousticness_%"],
        "Danceability vs Valence": corr.loc["danceability_%", "valence_%"],
        "Energy vs Valence": corr.loc["energy_%", "valence_%"]
    })

relationship_df = pd.DataFrame(relationship_results).round(3)

display(relationship_df)

relationship_df.to_csv(
    output_dir / "key_correlations_by_quartile.csv",
    index=False
)


The direction of these relationships stays the same, but some of them are stronger in Q4. For example, energy and acousticness go from about **-0.55 in Q1** to about **-0.67 in Q4**, and danceability and valence go from about **0.37** to about **0.50**.


### Correlation heatmaps: Q1 and Q4

I used the same scale on both plots so that the two groups are directly comparable.


In [ ]:
for label, corr in [
    ("Lowest 25% of streams", corr_q1),
    ("Highest 25% of streams", corr_q4)
]:
    plt.figure(figsize=(8, 6.5))
    plt.imshow(corr, vmin=-1, vmax=1, aspect="auto")

    plt.xticks(
        range(len(features)),
        features,
        rotation=45,
        ha="right"
    )
    plt.yticks(range(len(features)), features)

    plt.colorbar(label="Pearson correlation")
    plt.title(f"Audio-feature correlations: {label}")
    plt.tight_layout()
    plt.show()


## A closer look at two relationships

These scatterplots make the Q1 vs Q4 comparison easier to see.


In [ ]:
for q, short_label in [
    ("Q1: Lowest 25%", "Lowest 25%"),
    ("Q4: Highest 25%", "Highest 25%")
]:
    subset = df_analysis[df_analysis["stream_quartile"] == q]
    r = subset["energy_%"].corr(subset["acousticness_%"])

    plt.figure(figsize=(6.5, 4.5))
    plt.scatter(
        subset["acousticness_%"],
        subset["energy_%"],
        alpha=0.6
    )
    plt.xlabel("Acousticness (%)")
    plt.ylabel("Energy (%)")
    plt.title(f"Energy vs Acousticness: {short_label}\nr = {r:.2f}")
    plt.tight_layout()
    plt.show()


In [ ]:
for q, short_label in [
    ("Q1: Lowest 25%", "Lowest 25%"),
    ("Q4: Highest 25%", "Highest 25%")
]:
    subset = df_analysis[df_analysis["stream_quartile"] == q]
    r = subset["danceability_%"].corr(subset["valence_%"])

    plt.figure(figsize=(6.5, 4.5))
    plt.scatter(
        subset["danceability_%"],
        subset["valence_%"],
        alpha=0.6
    )
    plt.xlabel("Danceability (%)")
    plt.ylabel("Valence (%)")
    plt.title(f"Danceability vs Valence: {short_label}\nr = {r:.2f}")
    plt.tight_layout()
    plt.show()


## What I take from this

The main thing I see is that **most individual audio features do not change dramatically across the stream quartiles**. Speechiness is the clearest exception, and danceability also shows some evidence of a difference.

The correlation comparison is interesting because a few relationships become stronger in the highest-stream group. In particular:

- Energy vs acousticness: about **-0.55 in Q1** and **-0.67 in Q4**
- Danceability vs valence: about **0.37 in Q1** and **0.50 in Q4**

So, at least in this exploratory analysis, the difference between lower- and higher-streamed songs may show up partly in the **way the audio features relate to each other**, not just in the average value of each feature.

This is association only. I am not claiming that any of these audio features cause a song to get more streams.
